# Modelo-07 — Construção da Tabela Mestra do Sistema PSD

## Objetivo

Este notebook tem como objetivo integrar e padronizar os dados históricos do sistema de Portas de Plataforma (PSD), preparando uma tabela mestra para as etapas posteriores de processamento e treinamento do `CatBoostClassifier`.

A unidade de análise será o sistema PSD de cada **estação e plataforma** em uma determinada data. Dessa forma, os alarmes e as manutenções dos diferentes componentes — como PDMs, MDUs, sensores, controladores e portas de emergência — poderão representar conjuntamente o estado operacional daquele sistema.

Neste notebook serão realizadas somente a leitura, integração, padronização e auditoria das fontes. A criação das features e do alvo — ocorrência de uma falha nos 7 dias seguintes — será feita no notebook de processamento, evitando misturar a preparação dos eventos com a modelagem.

## Hipóteses que orientam a construção da tabela

A tabela mestra deve preservar os campos necessários para testar e transformar as evidências levantadas na Sprint 1 em features temporais. A inclusão de uma informação na tabela não significa que ela será obrigatoriamente utilizada pelo modelo.

| Hipótese | Resultado da investigação | Informação que deve ser preservada |
|---|---|---|
| H1 — padrões por horário | Confirmada | Horário do alarme, duração e identificação do PSD |
| H2 — regime semanal | Confirmada | Data do evento para derivar dia da semana e domingo |
| H3 — fluxo de passageiros | Refutada | Dados de catraca não serão integrados na primeira versão |
| H4 — aumento de alarmes antes da corretiva | Não confirmada | Tipo e horário dos alarmes serão preservados, sem assumir que a contagem sempre cresce antes da falha |
| H5 — tempo de normalização | Possui sinal, mas é insuficiente isoladamente | Início, normalização, duração e estação |
| H6 — posição central ou extremidade | Efeito prático pequeno | A posição da porta não será uma feature prioritária no modelo agregado por plataforma |
| H7 — predominância de falhas mecânicas | Sustentada | Causa, defeito, objeto e ação das manutenções históricas |

> **Prevenção de vazamento temporal:** causa, defeito e ação só poderão gerar features depois que a respectiva informação estiver disponível no histórico. Esses campos nunca poderão descrever a própria falha que o modelo está tentando prever.

In [2]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Semente: {RANDOM_SEED}")

Pandas: 3.0.5
NumPy: 2.5.1
Semente: 42
